# Salidas estructuradas: que el modelo rellene tu formulario

**Facsímil 4 · La caja de herramientas** — capítulo 2 (APIs de modelos: mensajes, *streaming* y salidas
estructuradas).

Si vas a meter lo que dice un LLM en una base de datos, en una factura o en una decisión, no puedes
permitirte un «más o menos». En este cuaderno dejas de tratar la respuesta del modelo como un texto del
que «sacar» datos a mano y empiezas a **exigirle un formato fijo** que validas automáticamente. Vas a
definir un esquema, comprobar una respuesta buena, **cazar** varias malas antes de que causen daño,
recoger **todos** los fallos de una vez, montar el bucle de reintento que usan los sistemas serios,
**cerrar** el esquema a campos inventados, ponerle **restricciones** y **reglas de negocio**, **anidar**
facturas con líneas, y ver el **contrato** (JSON Schema) que el modelo recibe. Las salidas estructuradas
convierten un texto simpático en un **dato fiable** o en un **error que se ve venir**.

### Qué vas a aprender
- Por qué «parsear» texto libre de un LLM es frágil, y qué lo resuelve.
- A definir un **esquema** (con Pydantic) que describe exactamente qué esperas y de qué tipo.
- A **validar** una respuesta y convertirla en un objeto con tipos (una fecha que es una fecha de verdad).
- A **rechazar** respuestas mal formadas con un mensaje claro, a recoger **varios errores** a la vez y a
  montar el **reintento** con el error.
- A **cerrar** el esquema (prohibir campos de más), poner **restricciones** (`Field`), **enumerados** y
  **reglas de negocio**, y a **anidar** modelos (una factura con varias líneas).
- A ver el **JSON Schema** que el modelo recibe como contrato, y por qué este patrón es la base de las
  *tools* de un agente (facsímil 5).

### Cómo se usa
Las celdas vienen **sin ejecutar**: pulsa ▶ de arriba abajo. No hace falta clave: simulamos las
respuestas del modelo.

### Cuánto cuesta
Unos 18 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
# En Colab:  !pip install pydantic
from pydantic import BaseModel, field_validator, ValidationError
from datetime import date

class Factura(BaseModel):
    proveedor: str
    importe_eur: float
    fecha: date
    concepto: str

    @field_validator("importe_eur")
    @classmethod
    def positivo(cls, v):
        if v <= 0:
            raise ValueError("el importe debe ser positivo")
        return v

print("Esquema definido. Una factura VÁLIDA debe tener: proveedor, importe>0, fecha y concepto.")


## 1. El problema de «parsear» texto libre

Imagina que le pides a un LLM los datos de una factura y responde: «La factura es de Acme, por unos 50
euros, creo que de finales de mayo». ¿Cómo extraes de ahí el importe exacto y la fecha para tu base de
datos? Con expresiones regulares frágiles que se rompen en cuanto el modelo cambia una coma. Es el
mismo error que cometía la informática antes de las APIs: tratar datos como texto. La solución: **pedir
estructura** (JSON con un esquema) y **validarla**. Así el modelo rellena un formulario, y tú compruebas
que está bien antes de usarlo.


## 2. El caso: extraer una factura de un correo

Esta sería una respuesta **buena** del modelo, en JSON. La validamos contra el esquema: si pasa, ya es
un objeto con **tipos correctos** —la fecha es un objeto fecha, no un texto—, listo para guardar u
operar (calcular el mes, comparar, sumar importes).


In [ ]:
respuesta_buena = '''{
  "proveedor": "Hosting Tarradellas S.L.",
  "importe_eur": 49.90,
  "fecha": "2026-05-31",
  "concepto": "Servidor dedicado mayo"
}'''

factura = Factura.model_validate_json(respuesta_buena)
print("Validada. Ya es un DATO, no un texto:")
print("  proveedor:", factura.proveedor)
print("  importe:", factura.importe_eur, "EUR (tipo", type(factura.importe_eur).__name__, ")")
print("  fecha:", factura.fecha, "-> tipo", type(factura.fecha).__name__)
print("  mes de la fecha (operable):", factura.fecha.month)
print("  trimestre:", (factura.fecha.month - 1)//3 + 1)


**De texto a dato.** Fíjate en lo que acaba de pasar: el `"2026-05-31"` que era una cadena de texto
ahora es un objeto fecha con el que se puede **operar** (sacar el mes, el trimestre, comparar). Eso es lo
que da un esquema: no «más o menos», sino un dato con el que tu código puede razonar.


## 3. Cuando el modelo se equivoca (y lo hará)

Los LLM, a veces, devuelven un importe como texto, se inventan un campo o se dejan otro. Sin validación,
eso entra en tu base de datos y revienta tres pasos después, lejos de donde se originó el problema. Con
esquema, **salta aquí mismo**, con un mensaje claro de qué campo falló y por qué. Probamos seis
respuestas defectuosas distintas.


In [ ]:
respuestas_malas = [
    ('importe como texto',   '{"proveedor": "Acme", "importe_eur": "cuarenta", "fecha": "2026-05-31", "concepto": "x"}'),
    ('importe negativo',     '{"proveedor": "Acme", "importe_eur": -10, "fecha": "2026-05-31", "concepto": "x"}'),
    ('importe cero',         '{"proveedor": "Acme", "importe_eur": 0, "fecha": "2026-05-31", "concepto": "x"}'),
    ('falta la fecha',       '{"proveedor": "Acme", "importe_eur": 10, "concepto": "x"}'),
    ('fecha mal formada',    '{"proveedor": "Acme", "importe_eur": 10, "fecha": "31 de mayo", "concepto": "x"}'),
    ('JSON roto',            '{"proveedor": "Acme", "importe_eur": 10, "fecha":'),
]
for motivo, r in respuestas_malas:
    try:
        Factura.model_validate_json(r); print(f"  [{motivo}] OK (no debería)")
    except ValidationError as e:
        err = e.errors()[0]
        campo = err['loc'][0] if err['loc'] else "(json)"
        print(f"  RECHAZADA ({motivo}) -> campo '{campo}': {err['msg']}")


**Eso es el valor del esquema.** Cada respuesta defectuosa se ha quedado en la puerta con un motivo
legible, *antes* de tocar nada: el importe que no es número, el negativo, el cero, el campo que falta, la
fecha mal escrita, incluso el JSON que ni siquiera cierra. Ninguna ha llegado a tu base de datos. El
modelo propone; tu código valida y dispone.


## 4. Todos los fallos de una vez

Cuando una respuesta tiene **varios** problemas, no quieres descubrirlos de uno en uno. Pydantic los
recoge **todos** en la misma pasada, así puedes devolvérselos juntos al modelo (o al usuario) en un solo
mensaje. Mandamos una respuesta a la que le faltan tres campos obligatorios y, encima, trae el importe
en negativo: el esquema los señala todos en la misma pasada.


In [ ]:
muy_mala = '{"importe_eur": -5}'   # falta proveedor, falta fecha, falta concepto, importe negativo
try:
    Factura.model_validate_json(muy_mala)
except ValidationError as e:
    print(f"La respuesta tiene {len(e.errors())} errores. Todos a la vez:")
    for err in e.errors():
        campo = err['loc'][0] if err['loc'] else "(json)"
        print(f"  - campo '{campo}': {err['msg']}")


## 5. El bucle de reintento

En un sistema real, si la validación falla, no te rindes: le **devuelves al modelo el error** y le pides
que lo corrija. Es el patrón estándar, porque los modelos fallan a veces pero suelen acertar al segundo
intento si les dices qué estuvo mal. Simulamos ese ida y vuelta.


In [ ]:
def pedir_al_modelo(intento, ultimo_error=None):
    # Simulación: el primer intento sale mal (coma decimal), el segundo corrige.
    if intento == 1:
        return '{"proveedor": "Acme", "importe_eur": "49,90", "fecha": "2026-05-31", "concepto": "x"}'
    return '{"proveedor": "Acme", "importe_eur": 49.90, "fecha": "2026-05-31", "concepto": "x"}'

ultimo_error = None
for intento in (1, 2, 3):
    try:
        f = Factura.model_validate_json(pedir_al_modelo(intento, ultimo_error))
        print(f"Intento {intento}: VÁLIDO -> {f.importe_eur} EUR. Guardado en base de datos.")
        break
    except ValidationError as e:
        ultimo_error = e.errors()[0]['msg']
        print(f"Intento {intento}: inválido ({ultimo_error}). Reintento devolviendo el error al modelo...")


## 6. Cerrar el esquema: rechazar lo que sobra

Por defecto, Pydantic **ignora** los campos de más: si el modelo se inventa un `descuento_secreto`, lo
tira sin avisar. A veces eso es justo lo que NO quieres: un campo inventado puede ser una alucinación que
prefieres detectar. Con `extra="forbid"`, el esquema **rechaza** cualquier campo que no esté en el
contrato.


In [ ]:
from pydantic import ConfigDict

class FacturaCerrada(Factura):
    model_config = ConfigDict(extra="forbid")

con_extra = '{"proveedor":"Acme","importe_eur":10,"fecha":"2026-05-31","concepto":"x","total_inventado":99}'
print("Esquema ABIERTO (por defecto): el campo de más se ignora en silencio")
print("   ->", Factura.model_validate_json(con_extra).proveedor, "(validó, campo extra descartado)")
print("\nEsquema CERRADO (extra='forbid'): el campo de más se rechaza")
try:
    FacturaCerrada.model_validate_json(con_extra)
except ValidationError as e:
    print("   -> RECHAZADA:", e.errors()[0]['msg'], "->", e.errors()[0]['loc'])


## 7. Restricciones y enumerados con `Field` y `Literal`

Un esquema puede poner **topes** y **listas cerradas** sin escribir validadores a mano. Con `Field`
exiges, por ejemplo, que el importe sea mayor que cero y no pase de 100.000 €. Con `Literal`, que la
moneda solo pueda ser una de tres. El modelo ya no puede colarte «euros» ni un importe disparatado.


In [ ]:
from pydantic import Field
from typing import Literal

class FacturaConTopes(BaseModel):
    proveedor: str = Field(min_length=1)
    importe_eur: float = Field(gt=0, le=100_000)        # >0 y <=100.000
    moneda: Literal["EUR", "USD", "GBP"]                # solo estos tres
    fecha: date

casos = [
    ('correcta',        '{"proveedor":"Acme","importe_eur":50,"moneda":"EUR","fecha":"2026-05-31"}'),
    ('importe enorme',  '{"proveedor":"Acme","importe_eur":250000,"moneda":"EUR","fecha":"2026-05-31"}'),
    ('moneda inválida', '{"proveedor":"Acme","importe_eur":50,"moneda":"euros","fecha":"2026-05-31"}'),
    ('proveedor vacío', '{"proveedor":"","importe_eur":50,"moneda":"EUR","fecha":"2026-05-31"}'),
]
for motivo, r in casos:
    try:
        FacturaConTopes.model_validate_json(r); print(f"  [{motivo}] OK")
    except ValidationError as e:
        print(f"  [{motivo}] RECHAZADA -> campo '{e.errors()[0]['loc'][0]}': {e.errors()[0]['msg']}")


## 8. Reglas de negocio: lo que es válido en *tu* sistema

Un JSON puede estar bien formado y aun así ser un dato imposible: una factura con fecha **futura**. Eso no
lo pilla un tipo ni un tope, hace falta una **regla de negocio**. La añadimos con un validador, junto a un
campo opcional con valor por defecto. Así el esquema se vuelve el contrato completo de lo que aceptas.


In [ ]:
from typing import Optional
class FacturaEstricta(Factura):
    iva_incluido: Optional[bool] = True       # opcional, por defecto True
    @field_validator("fecha")
    @classmethod
    def no_futura(cls, v):
        if v > date(2026, 6, 26):
            raise ValueError("la fecha no puede ser futura")
        return v

casos = [
    ('factura normal',  '{"proveedor":"Acme","importe_eur":10,"fecha":"2026-01-10","concepto":"x"}'),
    ('fecha futura',    '{"proveedor":"Acme","importe_eur":10,"fecha":"2027-01-10","concepto":"x"}'),
]
for motivo, r in casos:
    try:
        f = FacturaEstricta.model_validate_json(r)
        print(f"  [{motivo}] OK -> iva_incluido={f.iva_incluido} (valor por defecto aplicado)")
    except ValidationError as e:
        print(f"  [{motivo}] RECHAZADA -> {e.errors()[0]['msg']}")


## 9. Esquemas que anidan: una factura con varias líneas

El JSON anida, y los esquemas también. Una factura real tiene **líneas** (concepto, cantidad, precio), y
un total que debería ser la **suma** de las líneas. Modelamos eso con un esquema dentro de otro y una
regla que comprueba la coherencia del total: si el modelo se equivoca al sumar, salta.


In [ ]:
from pydantic import model_validator

class Linea(BaseModel):
    concepto: str
    cantidad: int = Field(gt=0)
    precio_unitario: float = Field(gt=0)

class FacturaConLineas(BaseModel):
    proveedor: str
    lineas: list[Linea]
    total_eur: float

    @model_validator(mode="after")
    def total_cuadra(self):
        suma = sum(l.cantidad * l.precio_unitario for l in self.lineas)
        if abs(suma - self.total_eur) > 0.01:
            raise ValueError(f"el total {self.total_eur} no cuadra con la suma de líneas {suma:.2f}")
        return self

buena = '{"proveedor":"Acme","lineas":[{"concepto":"Disco","cantidad":2,"precio_unitario":30.0},{"concepto":"Cable","cantidad":1,"precio_unitario":9.5}],"total_eur":69.5}'
mala  = '{"proveedor":"Acme","lineas":[{"concepto":"Disco","cantidad":2,"precio_unitario":30.0}],"total_eur":100.0}'
f = FacturaConLineas.model_validate_json(buena)
print(f"OK: {len(f.lineas)} líneas, total {f.total_eur} EUR (cuadra con la suma).")
try:
    FacturaConLineas.model_validate_json(mala)
except ValidationError as e:
    print("RECHAZADA:", e.errors()[0]['msg'])


**Lo que se ve.** El esquema no solo comprueba que cada línea tiene su forma; comprueba una relación
*entre* campos (el total = suma de líneas). Las reglas de negocio que cruzan varios campos viven en un
`model_validator`, y atrapan justo los errores que un humano cansado dejaría pasar.


## 10. El contrato que recibe el modelo: JSON Schema

Cuando pides «salida estructurada» a una API moderna, no le mandas tu clase de Python: le mandas un
**JSON Schema**, una descripción del formato que el modelo debe rellenar. Pydantic lo genera por ti. Esto
es exactamente lo que viaja a la API, y es el mismo mecanismo que define las *tools* de un agente.


In [ ]:
import json
esquema = Factura.model_json_schema()
print("JSON Schema que describe una Factura (esto es lo que ve el modelo):\n")
print(json.dumps(esquema, indent=2, ensure_ascii=False))


**El puente al facsímil 5.** Una *tool* de un agente es esto: un nombre, una descripción y un esquema
de argumentos. El modelo «llama» a la herramienta rellenando ese esquema, y tu código lo valida igual que
hemos validado la factura **antes** de ejecutar nada. Salidas estructuradas y herramientas son la misma
idea vista dos veces.


## 11. Sin esquema frente a con esquema: ¿dónde estalla el error?

El argumento decisivo no es elegancia, es **dónde** y **cuándo** te enteras del fallo. Sin esquema, el
dato malo entra y revienta pasos después, lejos del origen, difícil de rastrear. Con esquema, salta en la
puerta. Lo vemos con el mismo dato defectuoso (un importe con coma decimal escrito como texto).


In [ ]:
dato_malo = {"proveedor": "Acme", "importe_eur": "49,90", "fecha": "2026-05-31", "concepto": "x"}

print("SIN esquema: el dato entra crudo y el fallo aparece tarde, al operar con él")
try:
    registro = dict(dato_malo)              # paso 1: se guarda tal cual, sin mirar
    # ... pasos intermedios: viaja por el sistema sin que nadie lo revise ...
    iva = float(registro["importe_eur"]) * 0.21   # paso 3: aquí, lejos del origen, estalla
    print("   IVA:", iva)
except ValueError as e:
    print(f"   [CRASH tardío] {type(e).__name__}: {e}  <- lejos de donde entró el dato")

print("\nCON esquema: salta en la puerta, en el sitio, con el campo señalado")
try:
    Factura.model_validate(dato_malo)
except ValidationError as e:
    print(f"   [rechazo inmediato] campo '{e.errors()[0]['loc'][0]}': {e.errors()[0]['msg']}")


## 12. Pruébalo tú

1. **Importe máximo a tu gusto:** cambia el tope de `FacturaConTopes` a 1.000 € y comprueba qué pasa con
   una factura de 5.000. Los esquemas también ponen topes de negocio, no solo de tipo.
2. **Una moneda más:** añade `"CHF"` al `Literal` de la moneda. ¿Sigue rechazando «euros»?
3. **Líneas con descuento:** añade a `Linea` un campo opcional `descuento` (0 a 1) y haz que el total lo
   tenga en cuenta en el `model_validator`.
4. **Cierra el grifo:** aplica `extra="forbid"` a `FacturaConLineas` y mándale una línea con un campo
   inventado. ¿Lo caza?
5. **Lee el contrato:** genera el `model_json_schema()` de `FacturaConTopes` y localiza dónde aparece la
   lista cerrada de monedas y el tope del importe. Eso es lo que «entiende» el modelo.
6. **Rompe el reintento:** en la sección 5, haz que el modelo simulado falle siempre. ¿Qué debería hacer
   tu sistema tras 3 intentos? (Pista: no quedarse colgado para siempre.)


## 13. Errores comunes

- **Confiar en que el modelo «siempre» devuelve JSON válido.** No siempre. Valida, y ten un plan B
  (reintento, valor por defecto, error controlado).
- **Validar solo tipos y olvidar las reglas de negocio.** Un importe negativo, una fecha futura o un
  total que no cuadra son JSON válidos pero datos inválidos. El esquema debe capturar las tres cosas.
- **Dejar el esquema abierto sin querer.** Si un campo inventado debe ser sospechoso, usa `extra="forbid"`;
  si no, una alucinación entra y se descarta en silencio.
- **No devolver el error al modelo en el reintento.** Si solo reintentas sin decirle qué falló, repetirá
  el mismo fallo. El error es la pista para que corrija.
- **Reintentar sin tope.** Tres intentos y a un plan B; reintentar para siempre es colgar el sistema.
- **Parsear con expresiones regulares.** Frágil y propenso a romperse. Pide estructura y valida.


## 14. Qué te llevas

- Pedir **salida estructurada** (un esquema) convierte la respuesta del modelo en un **dato con tipos**,
  no en un texto que hay que interpretar a mano.
- La validación **caza los fallos en el sitio**, con un mensaje claro, y los recoge **todos** de una vez;
  puede **cerrar** el esquema, poner **topes** y **enumerados**, codificar **reglas de negocio** y
  **anidar** modelos. No solo tipos: el contrato completo de lo que aceptas.
- El **reintento con el error** es el patrón estándar (con tope de intentos): los modelos fallan, el
  sistema se recupera.
- El modelo recibe un **JSON Schema**, no tu clase; ese mismo mecanismo define las *tools* de un agente.

**Para seguir:** este patrón es la base de las *tools* de un agente (facsímil 5), donde cada herramienta
tiene un contrato de entrada y salida que se valida igual antes de ejecutar nada.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*